In [25]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

In [26]:
FILE = "flights_data.csv"

all_columns = pd.read_csv(FILE, nrows=0).columns
weather = [c for c in all_columns if c.startswith("origin_") or c.startswith("dest_")]
basic = ["MONTH", "DAY_OF_WEEK", "CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME_BLK", "CRS_ELAPSED_TIME",
         "OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DISTANCE", "DEP_DEL15", "CANCELLED",
         "MAKER", "MODEL", "SEATS", "PLANE_AGE"]

df = pd.read_csv(FILE, usecols=basic + weather)
print(df.shape)
display(df.head())

(3488373, 36)


,MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,DEP_DEL15,DEP_TIME_BLK,CRS_ARR_TIME,CANCELLED,CRS_ELAPSED_TIME,DISTANCE,origin_precipitation,origin_snowfall,origin_snow_depth,origin_temperature_2m,origin_wind_speed_10m,origin_wind_gusts_10m,origin_wind_direction_10m,origin_cloud_cover,origin_weather_code,origin_pressure_msl,dest_precipitation,dest_snowfall,dest_snow_depth,dest_temperature_2m,dest_wind_speed_10m,dest_wind_gusts_10m,dest_wind_direction_10m,dest_cloud_cover,dest_weather_code,dest_pressure_msl,MAKER,MODEL,SEATS,PLANE_AGE
0,1,4,AA,LAX,BOS,1358,1.0,1300-1359,2229,0.0,331.0,2611.0,0.2,0.0,0.00,17.3,2.5,8.9,236.0,79.0,51.0,1017.5,0.0,0.0,0.01,-8.8,12.9,25.9,274.0,0.0,0.0,1008.4,AIRBUS INDUSTRIE,A321-231,379.0,13.0
1,1,4,AA,JFK,LAX,941,1.0,0900-0959,1303,0.0,382.0,2475.0,0.0,0.0,0.01,-4.0,14.3,32.7,308.0,0.0,0.0,1006.4,0.2,0.0,0.00,17.3,2.5,8.9,236.0,79.0,51.0,1017.5,AIRBUS INDUSTRIE,A321-231,379.0,13.0
2,1,4,AA,LAX,BOS,2200,1.0,2200-2259,628,0.0,328.0,2611.0,0.0,0.0,0.00,13.3,2.9,5.8,53.0,83.0,3.0,1019.1,0.0,0.0,0.01,-8.9,7.9,15.9,246.0,50.0,2.0,1007.3,AIRBUS,A321-231,379.0,13.0
3,1,4,AA,JFK,LAX,1745,0.0,1700-1759,2100,0.0,375.0,2475.0,0.0,0.0,0.01,-4.8,10.3,24.9,305.0,1.0,0.0,1011.5,0.0,0.0,0.00,12.9,2.0,3.7,35.0,18.0,0.0,1018.8,AIRBUS,A321-231,379.0,13.0
4,1,4,AA,LAX,JFK,830,0.0,0800-0859,1654,0.0,324.0,2475.0,4.2,0.0,0.00,14.6,1.2,8.6,61.0,100.0,63.0,1016.5,0.0,0.0,0.01,-3.8,11.2,31.5,300.0,4.0,0.0,1010.1,AIRBUS,A321-231,379.0,13.0


In [27]:
missing = df.isna().mean() * 100
missing[missing > 0].sort_values(ascending=False).round(2)

PLANE_AGE                    3.91
DEP_DEL15                    2.12
SEATS                        0.58
MODEL                        0.58
MAKER                        0.58
origin_precipitation         0.49
origin_snow_depth            0.49
origin_temperature_2m        0.49
origin_wind_speed_10m        0.49
origin_wind_gusts_10m        0.49
origin_wind_direction_10m    0.49
origin_cloud_cover           0.49
origin_weather_code          0.49
origin_pressure_msl          0.49
origin_snowfall              0.49
dest_cloud_cover             0.49
dest_pressure_msl            0.49
dest_weather_code            0.49
dest_snow_depth              0.49
dest_wind_direction_10m      0.49
dest_wind_gusts_10m          0.49
dest_wind_speed_10m          0.49
dest_temperature_2m          0.49
dest_precipitation           0.49
dest_snowfall                0.49
CRS_ELAPSED_TIME             0.00
dtype: float64

In [28]:
TARGET = "DEP_DEL15"
df = df.dropna(subset=[TARGET])

dep = df["CRS_DEP_TIME"].astype(int)
arr = df["CRS_ARR_TIME"].astype(int)
df["dep_minute_of_day"] = (dep // 100 % 24) * 60 + dep % 100
df["arr_minute_of_day"] = (arr // 100 % 24) * 60 + arr % 100

# Plane info cleanup
df["MAKER"] = df["MAKER"].str.strip().str.split().str[0]   # "AIRBUS INDUSTRIE" -> "AIRBUS"
df["MODEL"] = df["MODEL"].str.strip()
df.loc[df["SEATS"] == 0, "SEATS"] = None

print("rows:", len(df), " positive rate:", round(df[TARGET].mean(), 3))

rows: 3414526  positive rate: 0.226


In [29]:
calendar = ["DAY_OF_WEEK"]
schedule = ["dep_minute_of_day", "arr_minute_of_day", "DEP_TIME_BLK", "CRS_ELAPSED_TIME"]
route    = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DISTANCE"]

FEATURES = calendar + schedule + route + weather
CATEGORICAL = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DEP_TIME_BLK"]

for col in CATEGORICAL:
    df[col] = df[col].astype("category")

print(len(FEATURES), "features")

29 features


In [31]:
THRESHOLD = 0.4

splits = [
    ([1],             2),   # train Jan        -> validate Feb
    ([1, 2],          3),   # train Jan-Feb    -> validate Mar
    ([1, 2, 3],       4),   # train Jan-Mar    -> validate Apr
    ([1, 2, 3, 4],    5),   # train Jan-Apr    -> validate May
    ([1, 2, 3, 4, 5], 6),   # train Jan-May    -> TEST on Jun
]

results = []
for train_months, valid_month in splits:
    train = df[df["MONTH"].isin(train_months)]
    valid = df[df["MONTH"] == valid_month]

    model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=63, verbose=-1)
    model.fit(train[FEATURES], train[TARGET])
    
    y_true = valid[TARGET]
    pred = model.predict_proba(valid[FEATURES])[:, 1]
    pred_label = (pred >= THRESHOLD).astype(int)
        
    results.append({
        "train":     train_months,
        "valid":     valid_month,
        "AUC":       roc_auc_score(y_true, pred),
        "precision": precision_score(y_true, pred_label),
        "recall":    recall_score(y_true, pred_label),
        "F1":        f1_score(y_true, pred_label),
    })
    print(f"month {valid_month} done")

pd.DataFrame(results)

month 2 done
month 3 done
month 4 done
month 5 done
month 6 done


,train,valid,AUC,precision,recall,F1
0,[1],2,0.660352,0.384991,0.185149,0.250047
1,"[1, 2]",3,0.684363,0.568243,0.106409,0.179252
2,"[1, 2, 3]",4,0.694804,0.432654,0.238998,0.307908
3,"[1, 2, 3, 4]",5,0.709472,0.509931,0.202338,0.289718
4,"[1, 2, 3, 4, 5]",6,0.740905,0.629901,0.242424,0.350106
